In [14]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical


In [15]:
def load_and_preprocess_data(data_dir, img_size=(224, 224)):
    """Carga y preprocesa imágenes desde el directorio de datos"""
    if not os.path.exists(data_dir):
        raise ValueError(f"El directorio {data_dir} no existe")
        
    X = []
    y = []
    
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    if not classes:
        raise ValueError(f"No se encontraron subdirectorios (clases) en {data_dir}")
    
    class_indices = {cls: i for i, cls in enumerate(classes)}
    
    print(f"Encontradas {len(classes)} clases: {classes}")
    
    total_images = 0
    for class_name in classes:
        class_dir = os.path.join(data_dir, class_name)
        class_idx = class_indices[class_name]
        

        image_files = [f for f in os.listdir(class_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        print(f"Procesando clase '{class_name}': {len(image_files)} imágenes encontradas")
        
        for img_file in image_files:
                
            img_path = os.path.join(class_dir, img_file)
            try:
        
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Error: No se pudo cargar la imagen {img_path}")
                    continue
                
            
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                img = cv2.resize(img, img_size)
                
                X.append(img)
                y.append(class_idx)
                total_images += 1

            except Exception as e:
                print(f"Error procesando {img_path}: {str(e)}")
    
    if total_images == 0:
        raise ValueError("No se pudieron cargar imágenes del directorio")
        
    print(f"Total de imágenes cargadas: {total_images}")
    
    
    X = np.array(X)
    y = np.array(y)

    print(f"Forma del array X: {X.shape}")
    print(f"Forma del array y: {y.shape}")
    

    X = X.astype('float32') / 255.0
    
    return X, y, classes

In [16]:

def split_dataset(X, y, num_classes, val_split=0.2, test_split=0.1):
    """Divide el dataset en train, validation y test"""
    if len(X) == 0 or len(y) == 0:
        raise ValueError("Los arrays X e y están vacíos")
        
    print(f"Dividiendo dataset de {len(X)} muestras en {num_classes} clases")
    print(f"Proporciones: train={1-val_split-test_split}, val={val_split}, test={test_split}")
    
    
    y_categorical = to_categorical(y, num_classes=num_classes)
    
    
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_categorical, 
        test_size=val_split + test_split,
        stratify=y_categorical, 
        random_state=42
    )
    print("\nPipeline de preprocesamiento completado con éxito!")

    test_ratio = test_split / (val_split + test_split)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, 
        test_size=test_ratio,
        stratify=y_temp, 
        random_state=42
    )
    
    print(f"División de datos:")
    print(f"- Train: {X_train.shape[0]} muestras")
    print(f"- Validación: {X_val.shape[0]} muestras")
    print(f"- Test: {X_test.shape[0]} muestras")
    
    
    return X_train, X_val, X_test, y_train, y_val, y_test


In [17]:
def create_data_generators(X_train, X_val, X_test, y_train, y_val, y_test, batch_size=32):
    """Crea generadores de datos con augmentation para entrenamiento"""
    print(f"Creando generadores de datos con batch_size={batch_size}")
    
    
    train_datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    
    
    val_datagen = ImageDataGenerator()
    
    train_generator = train_datagen.flow(
        X_train, y_train,
        batch_size=batch_size
    )
    
    val_generator = val_datagen.flow(
        X_val, y_val,
        batch_size=batch_size
    )
    
    test_generator = val_datagen.flow(
        X_test, y_test,
        batch_size=batch_size,
        shuffle=False
    )
    
    return train_generator, val_generator, test_generator


In [18]:
def preprocess_pipeline(data_dir, img_size=(224, 224), batch_size=32):
    """Pipeline completo de preprocesamiento de datos"""
        
    print(f"\nIniciando pipeline de preprocesamiento...")
    print(f"Directorio de datos: {data_dir}")
    print(f"Tamaño de imagen: {img_size}")
    print(f"Batch size: {batch_size}\n")

    X, y, classes = load_and_preprocess_data(data_dir, img_size)
    
    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
        X, y, num_classes=len(classes)
    )
    
    train_generator, val_generator, test_generator = create_data_generators(
        X_train, X_val, X_test, y_train, y_val, y_test, batch_size
    )
    print("\nPipeline de preprocesamiento completado con éxito!")

    return {
        'train_generator': train_generator,
        'val_generator': val_generator,
        'test_generator': test_generator,
        'classes': classes
    }

In [19]:
data_dir = "/Users/andres/proyecto_final/Real-Time-Spanish-Sign-Language-Recognition/datasets/SSLdictionary_augmented"
generators = preprocess_pipeline(data_dir, img_size=(224, 224), batch_size=32)


Iniciando pipeline de preprocesamiento...
Directorio de datos: /Users/andres/proyecto_final/Real-Time-Spanish-Sign-Language-Recognition/datasets/SSLdictionary_augmented
Tamaño de imagen: (224, 224)
Batch size: 32

Encontradas 19 clases: ['R', 'U', 'I', 'N', 'G', 'T', 'S', 'A', 'F', 'O', 'M', 'C', 'D', 'Q', 'E', 'B', 'K', 'L', 'P']
Procesando clase 'R': 270 imágenes encontradas


Premature end of JPEG file


Procesando clase 'U': 324 imágenes encontradas
Procesando clase 'I': 339 imágenes encontradas
Procesando clase 'N': 333 imágenes encontradas


Premature end of JPEG file
Premature end of JPEG file


Procesando clase 'G': 324 imágenes encontradas
Procesando clase 'T': 306 imágenes encontradas
Procesando clase 'S': 300 imágenes encontradas
Procesando clase 'A': 306 imágenes encontradas
Procesando clase 'F': 315 imágenes encontradas
Procesando clase 'O': 291 imágenes encontradas
Procesando clase 'M': 342 imágenes encontradas
Procesando clase 'C': 294 imágenes encontradas
Procesando clase 'D': 306 imágenes encontradas
Procesando clase 'Q': 360 imágenes encontradas
Procesando clase 'E': 309 imágenes encontradas
Procesando clase 'B': 285 imágenes encontradas
Procesando clase 'K': 324 imágenes encontradas
Procesando clase 'L': 336 imágenes encontradas
Procesando clase 'P': 330 imágenes encontradas
Total de imágenes cargadas: 5994
Forma del array X: (5994, 224, 224, 3)
Forma del array y: (5994,)
Dividiendo dataset de 5994 muestras en 19 clases
Proporciones: train=0.7000000000000001, val=0.2, test=0.1
División de datos:
- Train: 4195 muestras
- Validación: 1199 muestras
- Test: 600 muestra